# Asylum Data Audit (2010 to 2025)

## 1. Data check

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 1000)

DATA_DIR = Path('../data')
PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

raw_apps = pd.read_csv(DATA_DIR / 'asylum_applications' / 'asylum_applications.csv')
raw_decs = pd.read_csv(DATA_DIR / 'asylum_decisions' / 'asylum_decisions.csv')
raw_pop = pd.read_csv(DATA_DIR / 'asylum_population' / 'persons_of_concern.csv')
footnotes = pd.read_csv(DATA_DIR / 'asylum_applications' / 'footnotes.csv')

print(f'Applications : {raw_apps.shape}')
print(f'Decisions    : {raw_decs.shape}')
print(f'Population   : {raw_pop.shape}')
print(f'Footnotes    : {footnotes.shape}')

### Missing keys and data types

In [ ]:
# Defensive numeric coercion to handle asterisks or string types
app_num_cols = ['Applied']
for col in app_num_cols:
    raw_apps[col] = pd.to_numeric(raw_apps[col].astype(str).str.replace('*', '1', regex=False), errors='coerce').fillna(0)

dec_num_cols = ['dec total', 'Dec recognized', 'dec other', 'dec rejected', 'dec closed']
for col in dec_num_cols:
    raw_decs[col] = pd.to_numeric(raw_decs[col].astype(str).str.replace('*', '1', regex=False), errors='coerce').fillna(0)

pop_num_cols = ['Asylum-seekers']
for col in pop_num_cols:
    raw_pop[col] = pd.to_numeric(raw_pop[col].astype(str).str.replace('*', '1', regex=False), errors='coerce').fillna(0)

# Impute missing ISO codes for spatial mapping components
raw_apps['Country of Asylum ISO'] = raw_apps['Country of Asylum ISO'].fillna('UNK')
raw_apps['Country of Origin ISO'] = raw_apps['Country of Origin ISO'].fillna('UNK')
raw_decs['Country of Asylum ISO'] = raw_decs['Country of Asylum ISO'].fillna('UNK')
raw_decs['Country of Origin ISO'] = raw_decs['Country of Origin ISO'].fillna('UNK')
raw_pop['Country of Asylum ISO'] = raw_pop['Country of Asylum ISO'].fillna('UNK')
raw_pop['Country of Origin ISO'] = raw_pop['Country of Origin ISO'].fillna('UNK')

print('--- Missing ISO Codes after imputation ---')
print('Applications ISO nulls:', raw_apps[['Country of Asylum ISO', 'Country of Origin ISO']].isna().sum().to_dict())
print('Decisions ISO nulls   :', raw_decs[['Country of Asylum ISO', 'Country of Origin ISO']].isna().sum().to_dict())
print('Population ISO nulls  :', raw_pop[['Country of Asylum ISO', 'Country of Origin ISO']].isna().sum().to_dict())

print('\n--- Column Data Types ---')
print('Applied dtype        :', raw_apps['Applied'].dtype)
print('dec total dtype      :', raw_decs['dec total'].dtype)
print('Asylum-seekers dtype :', raw_pop['Asylum-seekers'].dtype)

### Decision breakdown versus reported totals

In [ ]:
part_sum = raw_decs['Dec recognized'] + raw_decs['dec other'] + raw_decs['dec rejected'] + raw_decs['dec closed']
mismatches = raw_decs[part_sum != raw_decs['dec total']].copy()
mismatches['gap'] = raw_decs['dec total'] - part_sum

print(f'Rows where parts do not equal total: {len(mismatches)}')
print('\nSample mismatch rows:')
print(mismatches[['Year', 'Country of Asylum', 'Country of Origin', 'Dec recognized', 'dec other', 'dec rejected', 'dec closed', 'dec total', 'gap']].head(5))

### Global volume by year

In [ ]:
yearly_apps = raw_apps.groupby('Year')['Applied'].sum()
yearly_decs = raw_decs.groupby('Year')['dec total'].sum()
yearly_pop = raw_pop.groupby('Year')['Asylum-seekers'].sum()

yearly = pd.DataFrame({
    'cases_filed': yearly_apps,
    'cases_decided': yearly_decs,
    'pending_cases': yearly_pop
})
yearly['clearance_rate_pct'] = (yearly['cases_decided'] / yearly['cases_filed']) * 100
yearly['annual_gap'] = yearly['cases_filed'] - yearly['cases_decided']

print(yearly)

### Takeaway
272 rows in the decisions file report a total that does not equal the sum of recognized, other, rejected, and closed claims. Switzerland in 2012, for example, reported total decisions without an outcome breakdown. We use `dec total` for overall volume and the individual outcome columns for grant and rejection rates.

## 2. Cases versus persons

### Countries reporting in cases

In [ ]:
cases_by_country = raw_apps[raw_apps['App_pc'] == 'C'].groupby('Country of Asylum').agg(
    rows=('Applied', 'count'),
    total_cases=('Applied', 'sum')
).sort_values('total_cases', ascending=False)

print(cases_by_country.head(8))

### Comparing US and Germany raw intake

In [ ]:
us_raw = raw_apps[raw_apps['Country of Asylum ISO'] == 'USA']['Applied'].sum()
de_raw = raw_apps[raw_apps['Country of Asylum ISO'] == 'DEU']['Applied'].sum()

print(f'Germany raw sum : {de_raw:,}')
print(f'US raw sum      : {us_raw:,}')

### Footnote conversion rules

In [ ]:
us_notes = footnotes[footnotes['Country of Asylum'].str.contains('United States', na=False)]
uk_notes = footnotes[footnotes['Country of Asylum'].str.contains('United Kingdom', na=False)]

for text in us_notes['Footnote'].unique()[:2]:
    print('US note:', text, '\n')

for text in uk_notes['Footnote'].unique()[:1]:
    print('UK note:', text, '\n')

### Applying US and UK multipliers

In [ ]:
def get_multiplier(row):
    if row['App_pc'] == 'C':
        if row['Country of Asylum ISO'] == 'USA':
            return 1.5
        elif row['Country of Asylum ISO'] == 'GBR':
            return 1.3
    return 1.0

raw_apps['multiplier'] = raw_apps.apply(get_multiplier, axis=1)
raw_apps['applications_filed_raw'] = raw_apps['Applied'].astype(int)
raw_apps['applications_filed_est'] = (raw_apps['applications_filed_raw'] * raw_apps['multiplier']).round().astype(int)

us_est = raw_apps[raw_apps['Country of Asylum ISO'] == 'USA']['applications_filed_est'].sum()
print(f'US raw claims        : {us_raw:,}')
print(f'US estimated persons : {us_est:,} (diff: +{us_est - us_raw:,})')

### Takeaway
25 countries report asylum intake in cases rather than persons. Germany counts every individual person, while the US counts cases that often include entire families. Applying UNHCR documented multipliers (1.5 for the US and 1.3 for the UK) adds 564,644 persons to the US intake, moving it closer to comparable scale with Germany.

## 3. Processing authority and procedure stages

### Government versus UNHCR processing

In [ ]:
authority_map = {'G': 'Government', 'U': 'UNHCR', 'J': 'Joint'}
raw_apps['authority'] = raw_apps['Procedure Type'].map(authority_map).fillna('Unknown')
raw_decs['authority'] = raw_decs['Procedure Type'].map(authority_map).fillna('Unknown')

by_auth = raw_apps.groupby(['Country of Asylum', 'authority'])['applications_filed_raw'].sum().unstack(fill_value=0)
by_auth['total'] = by_auth.sum(axis=1)
by_auth['unhcr_share'] = (by_auth.get('UNHCR', 0) / by_auth['total']) * 100

unhcr_heavy = by_auth[by_auth['total'] > 50000].sort_values('unhcr_share', ascending=False)
print(unhcr_heavy[['total', 'unhcr_share', 'Government', 'UNHCR']].head(8))

### Filtering by first instance stage

In [ ]:
print('Decision levels share globally:')
print(raw_apps['Decision level'].value_counts(normalize=True).map('{:.1%}'.format))

fi_only = raw_apps[raw_apps['Decision level'] == 'FI']
us_in_fi = (fi_only['Country of Asylum ISO'] == 'USA').sum()
de_in_fi = fi_only[fi_only['Country of Asylum ISO'] == 'DEU']['Applied'].sum()
de_total = raw_apps[raw_apps['Country of Asylum ISO'] == 'DEU']['Applied'].sum()

print(f'\nUS rows under FI filter      : {us_in_fi}')
print(f'Germany share under FI filter: {de_in_fi / de_total:.1%}')

### Takeaway
Filtering on `FI` drops the United States completely and removes most German claims. The US records affirmative claims as `IN` (USCIS) and defensive claims as `EO` (immigration court). Germany records claims under `NA` and `JR`. To avoid losing major countries, we keep all stages and track stage breakdown in a dedicated court stages table.

## 4. Decision outcomes and administrative closures

### Global outcome breakdown

In [ ]:
rec = raw_decs['Dec recognized'].sum()
oth = raw_decs['dec other'].sum()
rej = raw_decs['dec rejected'].sum()
clo = raw_decs['dec closed'].sum()
tot = raw_decs['dec total'].sum()

print(f'Total decisions : {tot:,}')
print(f'Refugee grants  : {rec:,} ({rec/tot*100:.1f}%)')
print(f'Temp protection : {oth:,} ({oth/tot*100:.1f}%)')
print(f'Rejected        : {rej:,} ({rej/tot*100:.1f}%)')
print(f'Closed no ruling: {clo:,} ({clo/tot*100:.1f}%)')

### Countries with high closure rates

In [ ]:
outcomes = raw_decs.groupby('Country of Asylum').agg(
    total=('dec total', 'sum'),
    closed=('dec closed', 'sum'),
    refugee=('Dec recognized', 'sum'),
    other=('dec other', 'sum')
)
outcomes = outcomes[outcomes['total'] > 100000].copy()
outcomes['closure_pct'] = (outcomes['closed'] / outcomes['total']) * 100
outcomes['refugee_pct'] = (outcomes['refugee'] / outcomes['total']) * 100
outcomes['other_pct'] = (outcomes['other'] / outcomes['total']) * 100

print(outcomes.sort_values('closure_pct', ascending=False)[['total', 'closure_pct', 'refugee_pct', 'other_pct']].head(8))

### South Africa permit cancellation footnote

In [ ]:
za_notes = footnotes[footnotes['Country of Asylum'].str.contains('South Africa', na=False)]
for text in za_notes['Footnote'].unique()[:2]:
    print('South Africa note:', text, '\n')

### Takeaway
One third of all global asylum decisions (33.6%) end without a ruling on the merits. Peru closed 92.0% of claims without a decision, and the US closed 57.6%. South Africa closure rate of 62.7% reflects an administrative purge in 2022, where 72,200 expired COVID permits were closed at once. In Spain, standard refugee recognition is 5.9%, while temporary protection reaches 40.9%, primarily for Venezuelan and Colombian applicants.

## 5. Intake, clearance, and backlogs

### Clearance rate and years to clear

In [ ]:
# Dynamic window using the most recent 2 reporting years
max_year = int(raw_decs['Year'].max())
recent_years = [max_year - 1, max_year]

app_sum = raw_apps.groupby('Country of Asylum')['applications_filed_raw'].sum()
dec_sum = raw_decs.groupby('Country of Asylum')['dec total'].sum()
pop_latest = raw_pop[raw_pop['Year'] == max_year].groupby('Country of Asylum')['Asylum-seekers'].sum()
recent_velocity = raw_decs[raw_decs['Year'].isin(recent_years)].groupby('Country of Asylum')['dec total'].sum() / len(recent_years)

balance = pd.DataFrame({
    'filed_total': app_sum,
    'decided_total': dec_sum,
    f'pending_{max_year}': pop_latest,
    'recent_annual_decisions': recent_velocity
}).dropna(subset=['filed_total']).fillna(0)
balance = balance[balance['filed_total'] > 500000].copy()
balance['clearance_rate_pct'] = (balance['decided_total'] / balance['filed_total']) * 100
balance['years_to_clear'] = balance[f'pending_{max_year}'] / balance['recent_annual_decisions'].replace(0, np.nan)

balance = balance.sort_values('filed_total', ascending=False)
print(f'Recent decision velocity window: {recent_years}')
print(balance[['filed_total', 'decided_total', 'clearance_rate_pct', f'pending_{max_year}', 'years_to_clear']].head(10))

### Balance equation check on Germany (2023)

In [ ]:
p_2022 = raw_pop[(raw_pop['Year'] == 2022) & (raw_pop['Country of Asylum ISO'] == 'DEU')]['Asylum-seekers'].sum()
p_2023 = raw_pop[(raw_pop['Year'] == 2023) & (raw_pop['Country of Asylum ISO'] == 'DEU')]['Asylum-seekers'].sum()
a_2023 = raw_apps[(raw_apps['Year'] == 2023) & (raw_apps['Country of Asylum ISO'] == 'DEU')]['Applied'].sum()
d_2023 = raw_decs[(raw_decs['Year'] == 2023) & (raw_decs['Country of Asylum ISO'] == 'DEU')]['dec total'].sum()

expected = p_2022 + a_2023 - d_2023
gap = p_2023 - expected

print(f'Starting backlog (2022) : {p_2022:,}')
print(f'+ Applied (2023)        : {a_2023:,}')
print(f'- Decided (2023)        : {d_2023:,}')
print(f'= Calculated backlog    : {expected:,}')
print(f'Reported backlog (2023) : {p_2023:,}')
print(f'Difference (gap)        : {gap:,}')

### Takeaway
Over the 16-year period, the US cleared 47.1% of filed claims and accumulated 3.72 million pending cases by 2025. At its recent processing pace, clearing this inventory would take 10.5 years without any new filings. Germany cleared 116.1% of claims over the same period, bringing its backlog down to 1.1 years of work. The Germany balance check shows a 21,296 case discrepancy in 2023 between reported stock and net flows, reflecting unrecorded departures, withdrawals, and annual inventory adjustments.

## 6. Clean table export

### Key cardinality check

In [ ]:
keys = ['Year', 'Country of Asylum ISO', 'Country of Origin ISO', 'Procedure Type']
multi_app = (raw_apps.groupby(keys).size() > 1).sum()
multi_dec = (raw_decs.groupby(keys).size() > 1).sum()
total_keys = len(raw_apps.groupby(keys))

print(f'Keys with multiple rows in Applications: {multi_app:,} ({multi_app/total_keys*100:.1f}%)')
print(f'Keys with multiple rows in Decisions   : {multi_dec:,} ({multi_dec/total_keys*100:.1f}%)')

### Aggregation and export

In [ ]:
# 1. Master corridor flows
corridor_keys = ['Year', 'Country of Asylum ISO', 'Country of Origin ISO', 'Country of Asylum', 'Country of Origin', 'authority']

apps_grouped = raw_apps.groupby(corridor_keys, as_index=False).agg({
    'applications_filed_raw': 'sum',
    'applications_filed_est': 'sum'
})

decs_grouped = raw_decs.groupby(corridor_keys, as_index=False).agg({
    'Dec recognized': 'sum',
    'dec other': 'sum',
    'dec rejected': 'sum',
    'dec closed': 'sum',
    'dec total': 'sum'
}).rename(columns={
    'Dec recognized': 'refugee_grants',
    'dec other': 'temporary_protection',
    'dec rejected': 'cases_rejected',
    'dec closed': 'closed_without_decision',
    'dec total': 'cases_decided'
})

annual_flows = pd.merge(apps_grouped, decs_grouped, on=corridor_keys, how='outer').fillna(0)
annual_flows = annual_flows.rename(columns={
    'Year': 'year',
    'Country of Asylum ISO': 'country_asylum_iso',
    'Country of Origin ISO': 'country_origin_iso',
    'Country of Asylum': 'country_asylum',
    'Country of Origin': 'country_origin'
})

count_cols = ['applications_filed_raw', 'applications_filed_est', 'cases_decided', 'refugee_grants', 'temporary_protection', 'cases_rejected', 'closed_without_decision']
for col in count_cols:
    annual_flows[col] = annual_flows[col].astype(int)

# Pre-calculated operational metrics
annual_flows['decision_rate_pct'] = np.where(
    annual_flows['applications_filed_est'] > 0,
    (annual_flows['cases_decided'] / annual_flows['applications_filed_est'] * 100).round(2),
    0.0
)
annual_flows['ghost_rate_pct'] = np.where(
    annual_flows['cases_decided'] > 0,
    (annual_flows['closed_without_decision'] / annual_flows['cases_decided'] * 100).round(2),
    0.0
)
annual_flows['recognition_rate_pct'] = np.where(
    annual_flows['cases_decided'] > 0,
    (annual_flows['refugee_grants'] / annual_flows['cases_decided'] * 100).round(2),
    0.0
)

# Export Parquet and web-ready JSON
annual_flows.to_parquet(PROCESSED_DIR / 'annual_asylum_flows.parquet', index=False)
annual_flows.to_json(PROCESSED_DIR / 'annual_asylum_flows.json', orient='records', indent=2)
print(f'Saved annual_asylum_flows.parquet and .json: {len(annual_flows):,} rows')

# 2. Pending inventory table
pending_clean = raw_pop.rename(columns={
    'Year': 'year',
    'Country of Asylum': 'country_asylum',
    'Country of Origin': 'country_origin',
    'Country of Asylum ISO': 'country_asylum_iso',
    'Country of Origin ISO': 'country_origin_iso',
    'Asylum-seekers': 'pending_cases_year_end'
})
pending_clean['pending_cases_year_end'] = pending_clean['pending_cases_year_end'].astype(int)
pending_clean.to_parquet(PROCESSED_DIR / 'pending_backlog_yearly.parquet', index=False)
print(f'Saved pending_backlog_yearly.parquet: {len(pending_clean):,} rows')

# 3. Court stages table
stages_apps = raw_apps.groupby(['Year', 'Country of Asylum ISO', 'Country of Asylum', 'Decision level'], as_index=False)['applications_filed_raw'].sum()
stages_decs = raw_decs.groupby(['Year', 'Country of Asylum ISO', 'Country of Asylum', 'Dec level'], as_index=False)['dec total'].sum()
court_stages = pd.merge(
    stages_apps.rename(columns={'Decision level': 'court_stage', 'applications_filed_raw': 'cases_filed'}),
    stages_decs.rename(columns={'Dec level': 'court_stage', 'dec total': 'cases_decided'}),
    on=['Year', 'Country of Asylum ISO', 'Country of Asylum', 'court_stage'],
    how='outer'
).fillna(0).rename(columns={'Year': 'year', 'Country of Asylum ISO': 'country_asylum_iso', 'Country of Asylum': 'country_asylum'})

court_stages['cases_filed'] = court_stages['cases_filed'].astype(int)
court_stages['cases_decided'] = court_stages['cases_decided'].astype(int)
court_stages.to_parquet(PROCESSED_DIR / 'court_stages_summary.parquet', index=False)
print(f'Saved court_stages_summary.parquet: {len(court_stages):,} rows')

### Verification

In [ ]:
print('--- Mart Verification ---')
df_flows = pd.read_parquet(PROCESSED_DIR / 'annual_asylum_flows.parquet')
df_pending = pd.read_parquet(PROCESSED_DIR / 'pending_backlog_yearly.parquet')
df_stages = pd.read_parquet(PROCESSED_DIR / 'court_stages_summary.parquet')

print(f'Flows   : {df_flows.shape}')
print(f'Pending : {df_pending.shape}')
print(f'Stages  : {df_stages.shape}')
print('\nSample flow row with pre-calculated rates:')
print(df_flows[['year', 'country_asylum_iso', 'country_origin_iso', 'applications_filed_est', 'cases_decided', 'decision_rate_pct', 'ghost_rate_pct']].head(3))

### Takeaway
Directly joining applications and decisions on country and year causes row duplication because both tables record multiple entries per corridor across different stages and dates. Pre-aggregating to corridor and authority before joining produces a clean 1:1 merge. The exports include pre-calculated decision, ghost, and recognition rates, along with a JSON export for direct frontend consumption.